# Análisis Exploratorio de Datos — Fase 3 y 3.5

Este notebook documenta el análisis exploratorio de los rendimientos de las
9 empresas colombianas, incluyendo la **imputación de valores faltantes con
Filtro de Kalman** (Fase 3.5). **No se ajusta ningún modelo econométrico ni de
machine learning todavía.**

El notebook es reproducible: reutiliza los módulos de `src/` y los datos
guardados en `datos/`. Si los datos aún no existen, se pueden generar ejecutando
`python -m src.pipeline` desde la raíz del proyecto.

Contenido:

1. Importación de configuración
2. Carga de datos
3. Validación básica
4. Precios
5. Rendimientos
6. Imputación de valores faltantes (Kalman)
7. Estadística descriptiva
8. Volatilidad
9. Correlaciones
10. Visualizaciones
11. Detección exploratoria de outliers
12. Conclusiones preliminares


In [ ]:
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en sys.path
RAIZ = Path().resolve()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (14, 5)
print("Raíz del proyecto:", RAIZ)
print("pandas", pd.__version__, "| numpy", np.__version__)


## 1. Importación de configuración

La configuración (tickers, fechas, frecuencia) vive en un único lugar: `config/settings.py`.


In [ ]:
from config.settings import (
    TICKERS, FECHA_INICIO, FECHA_FIN, FRECUENCIA, PRECIO_RENDIMIENTOS,
    DIAS_BURSATILES_ANIO,
)

print("Empresas seleccionadas:")
for nombre, ticker in TICKERS.items():
    print(f"  - {nombre:22s} {ticker}")
print("\nParámetros:")
print("  Fecha inicio:", FECHA_INICIO)
print("  Fecha fin (None = hoy):", FECHA_FIN)
print("  Frecuencia:", FRECUENCIA)
print("  Precio para rendimientos:", PRECIO_RENDIMIENTOS)


## 2. Carga de datos

Los datos crudos (sin modificar) se guardan en `datos/crudos/` como un CSV por empresa. Aquí se cargan y se organizan en un diccionario `{empresa: DataFrame}`.


In [ ]:
from src.data.data_manager import cargar_datos_crudos, nombre_archivo_crudo

crudos = cargar_datos_crudos()
print("Archivos crudos encontrados:", len(crudos))

# Construir un diccionario {empresa: DataFrame} usando el mapeo de tickers
datos_por_empresa = {}
for nombre, ticker in TICKERS.items():
    clave = nombre_archivo_crudo(ticker).replace(".csv", "")
    if clave in crudos:
        datos_por_empresa[nombre] = crudos[clave]

print("Empresas con datos:", list(datos_por_empresa.keys()))


## 3. Validación básica

Se ejecuta el validador de calidad, que revisa estructura, fechas, valores faltantes e inválidos. **No elimina** observaciones: solo las identifica.


In [ ]:
from src.data.data_validator import validar_varios

reporte = validar_varios(datos_por_empresa)
reporte[["empresa", "observaciones", "fecha_inicio", "fecha_fin",
         "faltantes", "duplicados", "precios_invalidos", "estado"]]


## 4. Precios

Se construye una tabla 'ancha' de precios ajustados (`Adj Close`) por empresa.


In [ ]:
from src.data.data_manager import _dataframe_precios_anchos

precios = _dataframe_precios_anchos(datos_por_empresa, PRECIO_RENDIMIENTOS)
precios.tail()


### Gráfico 1: precio de cada empresa a lo largo del tiempo


In [ ]:
precios.plot(subplots=True, figsize=(14, 20), layout=(5, 2),
             title="Precio ajustado por empresa (COP)", legend=False)
plt.tight_layout()
plt.show()


## 5. Rendimientos

Se calculan los rendimientos **simple** y **logarítmico**.

La primera observación de cada serie es `NaN` porque no hay precio previo; **no se elimina silenciosamente**, se conserva y se documenta.


In [ ]:
from src.preprocessing.returns import calcular_rendimientos

rend_simples, rend_log = calcular_rendimientos(precios)
print("Primeras filas de rendimientos logarítmicos (la 1ª observación es NaN):")
rend_log.head()


## 6. Imputación de valores faltantes mediante Filtro de Kalman

En la auditoría (Fase 3.5) se detectó **un único valor faltante** de precios:

- **Grupo Bolívar**, `Adj Close`, en `2026-03-10` (volumen 0, sin negociación).

Se decide imputarlo con un **modelo de nivel local** estimado con el Filtro de
Kalman y **suavizado** (smoothing), usando `statsmodels`. Solo se imputa la
variable `Adj Close` y el valor queda marcado con una bandera, sin tocar los
datos crudos.


In [ ]:
print("Valores faltantes en 'Adj Close' (por empresa):")
for nombre, df in datos_por_empresa.items():
    mascara = df[PRECIO_RENDIMIENTOS].isna()
    if mascara.any():
        print(f"  {nombre}: {int(mascara.sum())} faltante(s)")
        print(df.loc[mascara, ["Open", "High", "Low", "Close", "Adj Close", "Volume"]])


In [ ]:
from src.preprocessing.kalman_imputation import imputar_serie

serie = datos_por_empresa["Grupo Bolivar"][PRECIO_RENDIMIENTOS].copy()
imputada, bandera = imputar_serie(serie)

fechas_imp = serie.index[bandera]
for f in fechas_imp:
    print(f"Fecha imputada : {f.date()}")
    print(f"Valor imputado : {imputada.loc[f]:.4f}")


### Comparación visual (antes / después)


In [ ]:
ventana = pd.date_range(
    fechas_imp.min() - pd.Timedelta(days=15),
    fechas_imp.min() + pd.Timedelta(days=15),
)
orig = serie.reindex(ventana)
imp = imputada.reindex(ventana)

plt.figure(figsize=(10, 5))
plt.plot(imp.index, imp.values, label="Reconstruida (Kalman)", color="tab:blue")
plt.plot(orig.index, orig.values, label="Original (con hueco)", linestyle="--",
         alpha=0.7, color="tab:gray")
for f in fechas_imp:
    plt.scatter(f, imputada.loc[f], color="red", zorder=5, s=50, label="Valor imputado")
plt.title("Imputación Kalman — Grupo Bolívar (Adj Close)")
plt.ylabel("Adj Close (COP)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Validación artificial (MAE / RMSE)

Para evaluar la capacidad de reconstrucción del filtro se ocultan observaciones reales, se imputan y se comparan con el valor real.


In [ ]:
from src.preprocessing.kalman_imputation import validar_imputacion

serie_completa = serie.dropna()  # solo observados
r = validar_imputacion(serie_completa, n_ocultar=30, semilla=42)
print("Validación artificial (Grupo Bolívar):")
print(f"  n ocultados : {r['n_ocultados']}")
print(f"  MAE         : {r['mae']:.4f}")
print(f"  RMSE        : {r['rmse']:.4f}")
print(f"  MAE relativo: {r['mae_relativo']:.2%}")


In [ ]:
print("Reporte de imputaciones guardado:")
reporte_imp = pd.read_csv(RAIZ / "resultados" / "reporte_imputaciones_kalman.csv")
print(reporte_imp.to_string(index=False))

print("\nValidación por activo (resultados/validacion_kalman.csv):")
validacion = pd.read_csv(RAIZ / "resultados" / "validacion_kalman.csv")
print(validacion.round(4).to_string(index=False))


### Advertencia metodológica

El suavizado de Kalman usa información **posterior** al valor faltante. Esto es
válido para **reconstrucción histórica**, pero **NO** debe usarse directamente en
backtesting o predicción (sería data leakage). En fases posteriores, si se
requiere información en tiempo real, habrá que adaptar el procedimiento (usar
solo filtrado, no suavizado).


### Impacto antes / después de la imputación

La imputación de un único valor tiene un impacto despreciable en las estadísticas descriptivas (diferencias del orden de 1e-5).


In [ ]:
comparacion = pd.read_csv(RAIZ / "resultados" / "comparacion_antes_despues_kalman.csv", index_col=0)
comparacion.round(8)


## 7. Estadística descriptiva

Medidas de centro, dispersión, forma (asimetría y curtosis) y percentiles de los rendimientos (sobre datos imputados).


In [ ]:
from src.exploratory_analysis.descriptive_statistics import (
    estadisticas_descriptivas, resumir_volatilidad,
)
from src.data.data_manager import procesar_con_imputacion

# Reconstruir rendimientos sobre precios imputados (igual que el pipeline)
conjunto = procesar_con_imputacion(
    {n: {"dataframe": d, "ticker": TICKERS[n]} for n, d in datos_por_empresa.items()}
)
rend_log = conjunto["rend_log"]

estad = estadisticas_descriptivas(rend_log)
estad.round(6)


## 8. Volatilidad

Volatilidad diaria (desviación estándar) y anualizada. La anualización usa la convención $\sigma_{anual}=\sigma_{diaria}\sqrt{252}$, donde 252 es una aproximación al número de días bursátiles del año.


In [ ]:
vol = resumir_volatilidad(rend_log, dias=DIAS_BURSATILES_ANIO)
vol.round(4)


### Gráfico 4: volatilidad por activo


In [ ]:
vol["volatilidad_anualizada"].plot(kind="bar", color=sns.color_palette("viridis", 9))
plt.title("Volatilidad anualizada por activo")
plt.ylabel("Volatilidad anualizada")
plt.xlabel("Activo")
plt.show()


## 9. Correlaciones

Matriz de correlación de Pearson entre los rendimientos. Sirve para estudiar diversificación; **todavía no se usa para optimizar**.


In [ ]:
from src.exploratory_analysis.correlations import matriz_correlacion

corr = matriz_correlacion(rend_log)
corr.round(3)


### Gráfico 5: matriz de correlaciones


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación entre rendimientos logarítmicos")
plt.show()


## 10. Visualizaciones

Las versiones guardadas de los 5 gráficos (precios, rendimientos, distribución, volatilidad y correlación) se generan con `src/exploratory_analysis/plots.py` y quedan en `resultados/graficos/`.


## 11. Detección exploratoria de outliers

Se identifican observaciones extremas (z-score y percentiles) **sin eliminarlas**. Muchas coinciden con eventos reales de mercado (p. ej., marzo de 2020).


In [ ]:
from src.exploratory_analysis.outliers import detectar_outliers

outliers = detectar_outliers(rend_log)
print(f"Posibles outliers identificados: {len(outliers)}")
outliers.groupby("empresa").size()


## 12. Conclusiones preliminares

El siguiente resumen es **descriptivo** (no es una recomendación de inversión).


In [ ]:
from src.exploratory_analysis.resumen import (
    generar_resumen_descriptivo, imprimir_resumen,
)

resumen = generar_resumen_descriptivo(rend_log, matriz_corr=corr,
                                      reporte_calidad=reporte)
imprimir_resumen(resumen)


### Nota metodológica

- Los datos analizados son **diarios** desde `2020-01-01` hasta la fecha de ejecución.
- Los rendimientos se calcularon sobre el **precio ajustado** (`Adj Close`).
- Un único valor faltante (Grupo Bolívar, `2026-03-10`) fue imputado con Filtro de Kalman y queda marcado con bandera.
- Ninguna observación fue eliminada; los valores extremos solo se identificaron.
- Este análisis es la base para las fases siguientes (econometría, riesgo, optimización).
